# 05 - Content-Based

No product descriptions or images in this data, only `product_name`, `aisle`, and
`department`. Content-based here means building a TF-IDF representation of each product
from that text, then recommending products similar to what a user has already bought —
a user's profile is the average TF-IDF vector of their purchase history, and products are
ranked by cosine similarity to that profile.

Already-purchased items are excluded from the recommendations here, unlike the ALS model. 
That's a deliberate difference: content similarity is a discovery signal — "here's
something like what you buy that you haven't tried" — not a reorder predictor, so it's being
evaluated and used for what it's actually good at rather than competing with Personalized
Frequency on its own turf.

Logic in `src/content_based.py`.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd

from data_processing import load_raw_data
from eda import build_transactions
from baseline_models import get_eval_users, evaluate_model
from content_based import ContentBasedModel

## Load data

In [2]:
processed_path = Path.cwd().parent / "data" / "processed" / "transactions.parquet"

if processed_path.exists():
    txn = pd.read_parquet(processed_path)
else:
    data = load_raw_data()
    txn = build_transactions(data)

# content-based needs the raw products/aisles/departments tables directly
# (not just the flattened transactions) to build the text field per product
data = load_raw_data()

print(txn.shape)

(33819106, 15)


In [3]:
eval_users = get_eval_users(txn)
print("users with a labeled train basket:", len(eval_users))

users with a labeled train basket: 131209


## Fit

Building the TF-IDF matrix over `product_name + aisle + department`, and each user's
purchase-history profile from the prior set.

In [4]:
cb_model = ContentBasedModel(top_n=50).fit(txn, data)
print("tfidf matrix shape:", cb_model.tfidf_matrix.shape)
print("vocabulary size:", len(cb_model.vectorizer.vocabulary_))

tfidf matrix shape: (49688, 10624)
vocabulary size: 10624


In [5]:
sample_user = eval_users.iloc[0]["user_id"]
purchased = cb_model.user_purchases_.get(sample_user, [])
print(f"user {sample_user} purchase history (sample):", purchased[:5])
print(f"content-based recs:", cb_model.recommend(sample_user)[:10])

user 1 purchase history (sample): [196, 12427, 10258, 25133, 10326]
content-based recs: [8049, 25204, 24162, 8361, 25310, 16220, 38310, 14439, 34748, 5708]


## Evaluate

Same Precision@10 / Recall@10 harness as every other model. Excluding purchased items from
the recommendations means this is being scored on a genuinely harder version of the task —
it can only get credit for predicting *new* items in the next basket, not the reorders that
make up the majority of it. Worth keeping that in mind when comparing this number directly
against Personalized Frequency or ALS.

In [6]:
K = 10
cb_results = evaluate_model(lambda uid: cb_model.recommend(uid), eval_users, k=K)
cb_results

{'k': 10,
 'n_users_evaluated': 131209,
 'precision_at_k': 0.0027620056550998785,
 'recall_at_k': 0.002989471870799875}

In [7]:
pd.DataFrame([cb_results], index=["Content-Based"]).to_csv(
    Path.cwd().parent / "data" / "processed" / "content_based_results.csv"
)

## Overlap with ALS

Both this model and ALS are meant to serve as discovery signals rather than reorder
predictors, so the question that actually matters before combining them in the hybrid step
is whether they're finding the *same* new items through different paths, or genuinely
different ones. Measuring this with Jaccard similarity (intersection over union) between
each model's top-50 picks per user, averaged across a sample of eval users — 0 means
completely disjoint, 1 means identical.

In [8]:
from collaborative_filtering import ALSModel

als_model = ALSModel(factors=50, regularization=0.01, alpha=15.0, iterations=15).fit(txn)

sample_users = eval_users["user_id"].sample(n=2000, random_state=42).tolist()

jaccard_scores = []
for uid in sample_users:
    cb_recs = set(cb_model.recommend(uid))
    als_recs = set(als_model.recommend(uid, n=50))
    union = cb_recs | als_recs
    if union:
        jaccard_scores.append(len(cb_recs & als_recs) / len(union))

avg_jaccard = sum(jaccard_scores) / len(jaccard_scores) if jaccard_scores else 0
print(f"users compared: {len(jaccard_scores)}")
print(f"average Jaccard overlap: {avg_jaccard:.4f}")

C:\Users\shubh\AppData\Roaming\Python\Python314\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

users compared: 2000
average Jaccard overlap: 0.0246


In [9]:
pd.DataFrame(
    [{"n_users_compared": len(jaccard_scores), "avg_jaccard_overlap": avg_jaccard}]
).to_csv(Path.cwd().parent / "data" / "processed" / "als_content_overlap.csv", index=False)

### Reading this number

Average Jaccard overlap came out to 0.0246 — close to fully disjoint. ALS and content-based
are surfacing almost entirely different products: ALS is picking up collaborative signal
("users like you also bought this"), content-based is picking up textual/category similarity
("this looks like what you buy"), and the two barely agree on specific items. That's good
evidence for combining both in the hybrid step — each is covering ground the other misses,
rather than one being a redundant copy of the other.

### Findings

The content-based model extends the recommendation pipeline by using product metadata rather than customer-product interaction patterns. TF-IDF features were constructed from **product name, aisle, and department**, producing a matrix of **49,688 products and 10,624 vocabulary terms**. Each user's profile is represented by the average TF-IDF vector of their purchase history, with cosine similarity used to identify related products.

* **Content-based recommendations are primarily a discovery signal.** Previously purchased products are intentionally excluded, so the model focuses on recommending products that are similar to what a customer has purchased but has not yet tried. This is different from the Personalized Frequency baseline, which is optimized for predicting reorders.

* **Standalone predictive performance is low.** On the 131,209 evaluation users, the model achieves **Precision@10 = 0.00276** and **Recall@10 = 0.00299**. Because previously purchased items are excluded, the model is evaluated on the more difficult task of identifying genuinely new products rather than the repeat purchases that dominate the dataset.

* **The model provides a different signal from ALS.** The average Jaccard overlap between the top-50 recommendations from ALS and the content-based model is only **0.0246** across 2,000 sampled evaluation users. This indicates that the two models are recommending largely different products.

* **Low overlap is useful for the hybrid strategy.** ALS captures collaborative relationships based on shared purchasing behavior, while the content-based model captures similarity in product names and category structure. Their low recommendation overlap suggests that the models provide complementary rather than redundant discovery signals.

* **Modeling implication:** Content-based recommendations should not replace the strong reorder-oriented Personalized Frequency signal. Instead, they should be incorporated as an additional **discovery feature** alongside ALS, allowing the hybrid model to combine personalized reordering with complementary product-discovery signals.
